<a href="https://colab.research.google.com/github/chaiyawat19/DataScinceLab/blob/main/Lab11_DeepLearning_imageprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
import os.path

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import tensorflow as tf

from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
image_dir = Path('/content/drive/MyDrive/CP363107-67 Data Science for Marketing/lab10_Deep Learning/imageprocessing')
image_dir

In [ ]:
filepaths = list(image_dir.glob(r'**/*.jpg'))
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

filepaths = pd.Series(filepaths, name='Filepath').astype(str)
labels = pd.Series(labels, name='Label')

image_df = pd.concat([filepaths, labels], axis=1)

In [ ]:
image_df

In [ ]:
train_df, test_df = train_test_split(image_df, train_size=0.70, shuffle=True)

In [ ]:
train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    validation_split=0.1
)

test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    width_shift_range=0.2,
    height_shift_range=0.2,
    validation_split=0.1
)

In [ ]:
train_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=42,
    subset='training'
)

val_images = train_generator.flow_from_dataframe(
    dataframe=train_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=True,
    seed=42,
    subset='validation'
)

test_images = test_generator.flow_from_dataframe(
    dataframe=test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    class_mode='categorical',
    batch_size=32,
    shuffle=False

)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

# Define the CNN model
model = keras.Sequential(
    [
        keras.Input(shape=(224, 224, 3)),
        layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
        layers.MaxPooling2D(pool_size=(2, 2)),
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(2, activation="softmax"), # Output layer with softmax for 2 classes
    ]
)

# Compile the model
model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# Print the model summary
model.summary()

# Train the model
epochs = 10  # Adjust the number of epochs as needed
history = model.fit(train_images, validation_data=val_images, epochs=epochs)


# Evaluate the model on the test set
loss, accuracy = model.evaluate(test_images)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

# Make predictions
predictions = model.predict(test_images)
predicted_classes = np.argmax(predictions, axis=1)

# Generate a classification report
print(classification_report(test_images.classes, predicted_classes))

In [ ]:
predictions

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Predict the labels for the test set
predictions = model.predict(test_images)
predicted_labels = np.argmax(predictions, axis=1)

# Get true labels
true_labels = test_images.classes

# Create the confusion matrix
cm = confusion_matrix(true_labels, predicted_labels)

# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=test_images.class_indices,
            yticklabels=test_images.class_indices)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
image_dir = Path('/content/drive/MyDrive/CP363107-67 Data Science for Marketing/lab10_Deep Learning/unseen')

In [ ]:
filepaths = list(image_dir.glob(r'**/*.JPG'))
labels = list(map(lambda x: os.path.split(os.path.split(x)[0])[1], filepaths))

filepaths = pd.Series(filepaths, name='Filepath').astype(str)
labels = pd.Series(labels, name='Label')

image_df_unseen = pd.concat([filepaths, labels], axis=1)
print(image_df_unseen)

In [ ]:
test_generator_unseen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255
)
test_images_unseen = test_generator_unseen.flow_from_dataframe(
    dataframe=image_df_unseen,
    x_col='Filepath',
    y_col='Label',
    target_size=(224, 224),
    color_mode='rgb',
    batch_size=32
)

In [ ]:
predictions_unseen = (model.predict(test_images_unseen))
print(predictions_unseen)

# Iterate through each prediction in the array
for prediction in predictions_unseen:
  # Assuming prediction[0] is the probability for 'Healthy' and prediction[1] for 'Leafspot'
  if prediction[0] > prediction[1]:  # Check if 'Healthy' probability is higher
    print("Healthy")
  else:
    print("Leafspot")

In [ ]:
    # Display the image (optional)
    from PIL import Image # Import the Image module from PIL
    import matplotlib.pyplot as plt
    image_path = image_df_unseen['Filepath'].iloc[i]  # Get the image path
    img = Image.open(image_path)
    plt.imshow(img)
    plt.title(f"Prediction: {'Healthy' if prediction[0] > prediction[1] else 'Leafspot'}")
    plt.axis('off')
    plt.show()


In [ ]:
# save model
model.save('/content/plant_diease.keras')